# NIR White Lagkitan Corn (Three-Class)

In [ ]:
import pandas as pd
import numpy as np

## Load Data

In [ ]:
df_raw_data = pd.read_csv('appendix_raw_dataset1.csv')
df_raw_data

## Preprocessing

### SNV

In [ ]:
feature_cols = ['730nm', '760nm', '810nm', '860nm', '900nm', '940nm']

In [ ]:
df_snv = df_raw_data.copy()

def snv_row(x):
    x = x.astype(float)
    mean = x.mean()
    std = x.std(ddof=1)

    if std < 1e-8:
        return x * np.nan

    return (x - mean) / std

df_snv[feature_cols] = df_snv[feature_cols].apply(
    snv_row, axis=1, result_type='expand'
)

df_snv = df_snv.dropna()

## Data Preparation

### Selecting Features (X) and Targets (y)

In [ ]:
X = df_snv[feature_cols]
y = df_snv['Class'].map({'Bland': 0, 'Average': 1, 'Sweet': 2})

### Data Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

## Optuna Hyperparameter Tuning

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier

# For reproducibility
sampler = optuna.samplers.TPESampler(seed=42)
inner_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Tune to penalize instability
alpha = 0.5
stability_std_cap = 0.02

### Logistic Regression

In [ ]:
def objective_lr(trial):
    params = {
        'C': trial.suggest_float('C', 0.03, 3.0, log=True),

        'solver': 'lbfgs',

        'penalty': 'l2',

        'class_weight': trial.suggest_categorical(
            'class_weight',
            [None, 'balanced']
        ),

        'tol': trial.suggest_float('tol', 1e-5, 1e-4, log=True),
        'max_iter': 5000,
        'random_state': 42
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='f1_macro'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_lr = optuna.create_study(direction='maximize', sampler=sampler)
study_lr.optimize(objective_lr, n_trials=100)

study_lr.best_params, study_lr.best_value

### Random Forest

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 0.5, 0.8]
        ),
        'min_samples_split': trial.suggest_int('min_samples_split', 4, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 40),
        'class_weight': trial.suggest_categorical(
            'class_weight',
            [None, 'balanced']
        ),

        'bootstrap': True,
        'n_jobs': -1,
        'random_state': 42
    }

    model = Pipeline([
        ('clf', RandomForestClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='f1_macro'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * std_score

In [ ]:
study_rf = optuna.create_study(direction='maximize', sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

study_rf.best_params, study_rf.best_value

### Linear Discriminant Analysis

In [ ]:
def objective_lda(trial):
    params = {
    'solver': trial.suggest_categorical(
        'solver',
        ['lsqr', 'eigen']
    ),
    'shrinkage': trial.suggest_float(
        'shrinkage',
        0.01,
        0.5,
        log=True
    ),
    'store_covariance': False,
    'tol': trial.suggest_float(
        'tol',
        1e-4,
        1e-2,
        log=True
    )
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='f1_macro'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * std_score

In [ ]:
study_lda = optuna.create_study(direction='maximize', sampler=sampler)
study_lda.optimize(objective_lda, n_trials=100)

study_lda.best_params, study_lda.best_value

### Support Vector Machine

In [ ]:
def objective_svc(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 50.0, log=True),

        'kernel': trial.suggest_categorical(
            'kernel',
            ['linear', 'rbf']
        ),

        'gamma': trial.suggest_categorical(
            'gamma',
            ['scale', 'auto']
        ),

        'class_weight': trial.suggest_categorical(
            'class_weight',
            [None, 'balanced']
        ),

        'tol': trial.suggest_float(
            'tol',
            1e-5,
            1e-3,
            log=True
        ),

        'random_state': 42
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='f1_macro'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * std_score

In [ ]:
study_svc = optuna.create_study(direction='maximize', sampler=sampler)
study_svc.optimize(objective_svc, n_trials=100)

study_svc.best_params, study_svc.best_value

### XGBoost

In [ ]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 4),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.1,
            log=True
        ),

        'n_estimators': trial.suggest_int('n_estimators', 100, 300),

        'min_child_weight': trial.suggest_float(
            'min_child_weight',
            1.0,
            10.0
        ),

        'gamma': trial.suggest_float(
            'gamma',
            0.0,
            0.5
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.7,
            1.0
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            0.7,
            1.0
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda',
            0.5,
            10.0,
            log=True
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha',
            0.0,
            0.5
        ),

        'max_delta_step': trial.suggest_int(
            'max_delta_step',
            0,
            5
        ),

        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1
    }

    model = Pipeline([
        ('clf', XGBClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='f1_macro'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * std_score

In [ ]:
study_xgb = optuna.create_study(direction='maximize', sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=100)

study_xgb.best_params, study_xgb.best_value

## Validation Set Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
models_val = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**study_lr.best_params)),
    ]),

    'Random Forest': RandomForestClassifier(**study_rf.best_params),

    'LDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**study_lda.best_params)),
    ]),

    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**study_svc.best_params)),
    ]),

    'XGBoost': XGBClassifier(**study_xgb.best_params)
}

In [ ]:
val_predictions = {}
val_metrics = {}
val_reports ={}

for model_name, model in models_val.items():

    model.fit(X_train, y_train)

    val_predictions[model_name] = model.predict(X_val)

    val_metrics[model_name] = {
        'Accuracy': accuracy_score(y_val, val_predictions[model_name]),
        'Precision': precision_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'Recall': recall_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'F1': f1_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
    }

    val_reports[model_name] = classification_report(y_val, val_predictions[model_name], zero_division=0)

### Summary Metrics

In [ ]:
df_val_metrics = pd.DataFrame(val_metrics)
df_val_metrics.T.sort_values(by='Accuracy', ascending=False)

### Classification Report

In [ ]:
for model_names, reports in val_reports.items():
    print(f'{model_names}:\n {reports} \n -------------------------------------------------------')

## Test Set Metrics

In [ ]:
X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)

In [ ]:
final_model = XGBClassifier(**study_xgb.best_params)
final_model.fit(X_combined, y_combined)

In [ ]:
test_predictions = final_model.predict(X_test)

test_metrics = {
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, test_predictions),
    'Precision': precision_score(y_test, test_predictions, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, test_predictions, average='weighted', zero_division=0),
    'F1': f1_score(y_test, test_predictions, average='weighted', zero_division=0),
}

### Summary Metrics

In [ ]:
df_test_metrics = pd.DataFrame([test_metrics])
df_test_metrics

### Classification Report

In [ ]:
print(classification_report(y_test, test_predictions))